In [1]:
import pandas as pd

In [2]:
collection = pd.read_csv('collection.csv')
topics = pd.read_csv('topics.csv')

In [3]:
# verify if csv files have been read correctly
print(collection.shape)
print(collection.head())
print(topics.shape)
print(topics.head())

(120, 2)
  passage_id                                            passage
0        P01  The first two years of these two programs are ...
1        P02  In the initial two years, the Software Enginee...
2        P03  The first two years of both Software Engineeri...
3        P04  The Software Engineering and Computer Science ...
4        P05  The initial two years in both Software Enginee...
(106, 4)
  topic_id                                              topic question_id  \
0      W01  what's the difference between CS and SE programs?      W01Q01   
1      W01  what's the difference between CS and SE programs?      W01Q02   
2      W01  what's the difference between CS and SE programs?      W01Q03   
3      W01  what's the difference between CS and SE programs?      W01Q04   
4      W02                What choice of electives available?      W02Q01   

                                            question  
0      What sets apart CS programs from SE programs?  
1  How do CS and SE progr

In [4]:
# build bm25 index over FAQ passage collection
from rank_bm25 import BM25Okapi

passages = collection['passage'].tolist()
tokenised_passages = [p.lower().split() for p in passages]

bm25 = BM25Okapi(tokenised_passages)

In [5]:
query = topics['question'].iloc[0]
print("Query:", query)

tokenised_query = query.lower().split()
scores = bm25.get_scores(tokenised_query)

top_k = 3
top_indices = scores.argsort()[::-1][:top_k]

for ind in top_indices:
    print(f"\nScore: {scores[ind]:.2f}")
    print(f"Passage ID: {collection.iloc[ind]['passage_id']}")
    print(f"Passage: {collection.iloc[ind]['passage'][:200]}")

Query: What sets apart CS programs from SE programs?

Score: 9.35
Passage ID: P35
Passage: Yes, all programs, apart from the Data Science degree, have secured professional-level accreditation from ACS.

Score: 6.41
Passage ID: P01
Passage: The first two years of these two programs are near-identical. Then, in the third year of the SE degree, students spend a whole year (roughly 40 working weeks) in an industry placement. They work in a 

Score: 5.66
Passage ID: P67
Passage: There is no direct answer. It is easy between some programs, and not as easy between some programs. For example, within first and second year, it is easy to transfer between SE and CS programs. Also, 


In [6]:
# Load qrels.txt from Walert to evaluate retrieval qualtiy
qrels = pd.read_csv('qrels.txt', sep='\t', names=['query_id', 'unused', 'passage_id', 'relevance'])

print(qrels.head())

  query_id  unused passage_id  relevance
0   W01Q01       0        P01          2
1   W01Q01       0        P02          2
2   W01Q01       0        P03          2
3   W01Q01       0        P04          2
4   W01Q01       0        P05          2


In [7]:
# checks if a relevant passage appears in top-3 retrieved for each question
def retrieve_top_k(question, k=3):
    tokenised_query = question.lower().split()
    scores = bm25.get_scores(tokenised_query)
    top_indices = scores.argsort()[::-1][:k]
    return [collection.iloc[ind]['passage_id'] for ind in top_indices]

relevant_passages = qrels.groupby('query_id')['passage_id'].apply(set).to_dict()

k = 3
hits = 0
total = 0

for _, row in topics.iterrows():
    q_id = row['question_id']
    question = row['question']

    if q_id not in relevant_passages:
        continue

    retrieved = set(retrieve_top_k(question, k=k))
    relevant = relevant_passages[q_id]

    if retrieved & relevant:
        hits += 1
    total += 1

accuracy = hits / total
print(f"Top-{k} retrieval accuracy: {accuracy:.2%} ({hits}/{total} questions)")

Top-3 retrieval accuracy: 67.71% (65/96 questions)


In [8]:
import ollama

# Following Walert's prompt structure, combine retrieval + generation using Ollama
def build_prompt(question, context_passages):
    prompt = (
        "Generate an answer based on the retrieved documents for the following question. "
        "If the retrieved documents are not related to the question, answer NA.\n\n"
        f"Question: {question}\n"
        f"Document 1: {context_passages[0]}\n"
        f"Document 2: {context_passages[1]}\n"
        f"Document 3: {context_passages[2]}\n"
        "Answer: "
    )
    return prompt

def get_passages(question, k=3):
    tokenised_query = question.lower().split()
    scores = bm25.get_scores(tokenised_query)
    top_indices = scores.argsort()[::-1][:k]
    return [collection.iloc[ind]['passage'] for ind in top_indices]

def generate_answer(question, model='llama3.2:3b'):
    context_passages = get_passages(question)
    prompt = build_prompt(question, context_passages)
    response = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    return response['message']['content'], context_passages

In [9]:
sample_questions = topics['question'].iloc[:5].tolist()

for q in sample_questions:
    answer, passages = generate_answer(q)
    print(f"Question: {q}")
    print(f"Generated Answer: {answer}")
    print("-"*80)

Question: What sets apart CS programs from SE programs?
Generated Answer: NA, as Document 1 only mentions accreditation and does not compare CS and SE programs, and Document 2 explains the differences in the last two years of the programs, not what sets them apart. Document 3 discusses transfer requirements between some programs, but does not provide a direct answer to what sets apart CS programs from SE programs.
--------------------------------------------------------------------------------
Question: How do CS and SE programs differ from each other?
Generated Answer: Based on the retrieved documents, the CS and SE programs differ from each other in the following ways:

* In the third year of the SE degree, students spend a whole year in an industry placement, working in a software firm and engaging in all facets of software development process.
* In the final year (third in CS and fourth in SE), the projects and electives differ. SE students do a large in-house project and more SE e